# Come gli errori nel dataset di training impattano i modelli di Machine Learning

**Architetture Dati — Data Quality**

In questo notebook prendo un dataset, lo "sporco" in modo controllato e progressivo, e guardo come
peggiorano i modelli di Machine Learning e perche'. Parto dal dataset *Breast Cancer Wisconsin*
(lo stesso della baseline `breast_cancer.ipynb`) e uso i due modelli gia' visti: **Decision Tree** e **SVM**.

Seguendo le slide di Data Quality, considero i 4 tipi di errore tipici:

1. **Noisy labels** — etichette sbagliate (diagnosi M/B invertita)
2. **Outliers** — valori anomali fuori scala
3. **Missing values** — valori mancanti
4. **Overlapping** — classi che si sovrappongono per il rumore sulle feature

Per ognuno: introduco l'errore a livelli crescenti (dal 10% al 40%), misuro il calo delle performance e provo una
tecnica di pulizia per limitarne l'effetto.

**Regola importante:** sporco solo i dati di **training**. Il **test set resta pulito**, perche'
rappresenta i dati "veri" su cui voglio che il modello funzioni.


## I 4 errori e le dimensioni di qualita' dei dati (slide del corso)

Le slide "Data Quality for Machine Learning" collegano ogni tipo di errore a una **dimensione di qualita'**
violata. Questa corrispondenza e' il filo conduttore del notebook:

| Errore | Dimensione di qualita' violata (slide) |
|---|---|
| Noisy labels (etichette sbagliate) | Accuratezza semantica / Consistency |
| Outliers (valori anomali) | Accuratezza sintattica |
| Missing values (valori mancanti) | Completezza |
| Overlapping data points | Consistency |

In pratica: sporcare i dati significa peggiorare una di queste dimensioni; le tecniche di pulizia servono a
recuperarla.

In [ ]:
# === LIBRERIE E DATI ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, recall_score, f1_score

SEED = 42

# Carico il dataset (UCI Breast Cancer Wisconsin Diagnostic)
data = fetch_ucirepo(id=17)
X = data.data.features.reset_index(drop=True)
y = data.data.targets["Diagnosis"].map({"M": 1, "B": 0}).reset_index(drop=True)
print("Dataset:", X.shape, "- Maligni:", int(y.sum()), "Benigni:", int((y == 0).sum()))

# Divido in training e test. Sporchero' SOLO il training; il test resta pulito.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
X_train = X_train.reset_index(drop=True); y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True);   y_test = y_test.reset_index(drop=True)
print("Training:", X_train.shape, "- Test (pulito):", X_test.shape)

# Salvo training come array + media e deviazione standard di ogni colonna (mi servono per il rumore)
X_train_np = X_train.values.astype(float)
y_train_np = y_train.values.astype(int)
COL_MEAN = X_train.mean().values
COL_STD = X_train.std().replace(0, 1e-9).values

def valuta(modello):
    """Addestra gia' fatto: calcola le metriche sul test PULITO. Classe positiva = Maligno (1)."""
    pred = modello.predict(X_test.values)  # .values: stesso formato del training (niente warning)
    return {
        "accuracy": accuracy_score(y_test, pred),
        "recall": recall_score(y_test, pred, pos_label=1, zero_division=0),
        "f1": f1_score(y_test, pred, pos_label=1, zero_division=0),
    }

## Come sporco i dati

Scrivo una funzione per ogni tipo di errore. Ognuna lavora su una copia del training e usa un seme fisso,
cosi' i risultati sono sempre uguali se rieseguo. Il livello dice quanto rumore aggiungo (es. 0.30 = 30%).

I due modelli li costruisco come "pipeline": una catena di passaggi. Metto sempre un riempimento dei valori
mancanti con la media (serve solo nell'esperimento sui missing, altrimenti non fa nulla). L'SVM in piu' ha
lo `StandardScaler` perche' e' sensibile alla scala dei numeri.


In [ ]:
# === FUNZIONI CHE SPORCANO I DATI ===
def rumore_label(livello, seed=SEED):
    """Etichette errate: inverte una frazione 'livello' di diagnosi (M<->B)."""
    np.random.seed(seed)
    y_sporco = y_train_np.copy()
    n = int(round(livello * len(y_sporco)))
    idx = np.random.choice(len(y_sporco), size=n, replace=False)
    y_sporco[idx] = 1 - y_sporco[idx]
    return X_train_np.copy(), y_sporco

def rumore_outlier(livello, seed=SEED):
    """Outlier: mette valori estremi (media +/- 6 deviazioni standard) in una frazione di celle."""
    np.random.seed(seed)
    X_sporco = X_train_np.copy()
    n = int(round(livello * X_sporco.size))
    righe = np.random.randint(0, X_sporco.shape[0], n)
    colonne = np.random.randint(0, X_sporco.shape[1], n)
    segno = np.random.choice([-1, 1], n)
    X_sporco[righe, colonne] = COL_MEAN[colonne] + segno * 6 * COL_STD[colonne]
    return X_sporco, y_train_np.copy()

def rumore_missing(livello, seed=SEED):
    """Valori mancanti: mette NaN in una frazione di celle."""
    np.random.seed(seed)
    X_sporco = X_train_np.copy()
    n = int(round(livello * X_sporco.size))
    righe = np.random.randint(0, X_sporco.shape[0], n)
    colonne = np.random.randint(0, X_sporco.shape[1], n)
    X_sporco[righe, colonne] = np.nan
    return X_sporco, y_train_np.copy()

def rumore_overlap(sigma, seed=SEED):
    """Overlapping: aggiunge rumore gaussiano alle feature, le classi si sovrappongono."""
    np.random.seed(seed)
    X_sporco = X_train_np.copy()
    X_sporco = X_sporco + np.random.normal(0, 1, X_sporco.shape) * COL_STD * sigma
    return X_sporco, y_train_np.copy()

# === I DUE MODELLI ===
def crea_albero():
    return Pipeline([("riempi", SimpleImputer(strategy="mean")),
                     ("modello", DecisionTreeClassifier(max_depth=3, random_state=SEED))])

def crea_svm():
    return Pipeline([("riempi", SimpleImputer(strategy="mean")),
                     ("scala", StandardScaler()),
                     ("modello", SVC(kernel="rbf", random_state=SEED))])

MODELLI = {"Decision Tree": crea_albero, "SVM": crea_svm}

def prova_su_livelli(funzione_rumore, livelli, nome_x):
    """Per ogni livello: addestra i modelli sui dati sporchi e li valuta sul test pulito."""
    risultati = []
    for liv in livelli:
        X_sporco, y_sporco = funzione_rumore(liv)
        for nome, crea in MODELLI.items():
            m = crea(); m.fit(X_sporco, y_sporco)
            voti = valuta(m)
            voti[nome_x] = liv; voti["modello"] = nome
            risultati.append(voti)
    return pd.DataFrame(risultati)

def grafico(df, nome_x, titolo):
    fig, assi = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, metrica in zip(assi, ["accuracy", "recall", "f1"]):
        for nome in MODELLI:
            sub = df[df["modello"] == nome]
            ax.plot(sub[nome_x], sub[metrica], marker="o", label=nome)
        ax.set_title(metrica); ax.set_xlabel(nome_x); ax.set_ylim(0, 1.02); ax.legend()
    fig.suptitle(titolo)
    plt.tight_layout(); plt.show()

# Livelli uguali per i primi tre esperimenti
LIVELLI = [0.0, 0.1, 0.2, 0.3, 0.4]

## Baseline: i modelli sul dataset pulito

Prima di sporcare i dati, addestro i due modelli sul dataset **originale (pulito)** e misuro le performance
sul test. Questi numeri sono il **riferimento**: in ogni esperimento successivo confronto i modelli
addestrati sui dati sporchi con questa baseline, per vedere quanto peggiorano. (Nei grafici, il livello di
rumore 0 corrisponde esattamente a questa baseline.)

In [ ]:
# === BASELINE: MODELLI SUL DATASET PULITO (riferimento) ===
print("Performance sul dataset PULITO (nessun errore introdotto):")
righe = []
for nome, crea in MODELLI.items():
    m = crea(); m.fit(X_train_np, y_train_np)
    righe.append({"modello": nome, **valuta(m)})
baseline = pd.DataFrame(righe)
display(baseline.round(3))

baseline.set_index("modello")[["accuracy", "recall", "f1"]].plot(
    kind="bar", figsize=(7, 4.5), title="Baseline: performance sui dati puliti")
plt.ylim(0, 1.02); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## Cross-validation: una stima piu' affidabile della baseline

Un singolo split train/test puo' capitare "fortunato" o "sfortunato". La **cross-validation** divide il
training in piu' parti (qui 5), addestra e valuta piu' volte e ne fa la media: e' una stima piu' stabile
delle performance. La calcolo sul dataset pulito come riferimento.

In [ ]:
# === CROSS-VALIDATION SUL DATASET PULITO ===
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for nome, crea in MODELLI.items():
    punteggi = cross_val_score(crea(), X_train_np, y_train_np, cv=cv, scoring="accuracy")
    print(f"{nome:14} accuracy media = {punteggi.mean():.3f} (+/- {punteggi.std():.3f})")

## 1. Noisy labels (etichette sbagliate)

*Dimensione di qualita' violata: accuratezza semantica / consistency.*

Inverto la diagnosi (Maligno/Benigno) di una parte dei pazienti nel training. E' di solito l'errore piu'
dannoso, perche' confonde direttamente cio' che il modello deve imparare.


In [ ]:
df_label = prova_su_livelli(rumore_label, LIVELLI, "livello")
display(df_label.round(3))
grafico(df_label, "livello", "Effetto delle etichette sbagliate (test pulito)")

**Pulizia:** addestro un albero semplice sui dati sporchi e tengo solo le righe la cui etichetta
coincide con quella prevista dal modello; quelle "incoerenti" probabilmente sono errori e le tolgo prima di
riaddestrare. E' l'idea di pulire i dati *prima* del training.


In [ ]:
def pulisci_label(X_sporco, y_sporco):
    m = crea_albero(); m.fit(X_sporco, y_sporco)
    previsto = m.predict(X_sporco)
    tieni = previsto == y_sporco          # tengo solo le righe coerenti
    return X_sporco[tieni], y_sporco[tieni], tieni

righe = []
for liv in LIVELLI:
    X_sporco, y_sporco = rumore_label(liv)
    m = crea_svm(); m.fit(X_sporco, y_sporco)
    righe.append({"livello": liv, "dati": "Sporchi", **valuta(m)})
    Xp, yp, tieni = pulisci_label(X_sporco, y_sporco)
    m = crea_svm(); m.fit(Xp, yp)
    righe.append({"livello": liv, "dati": "Puliti", **valuta(m)})
df_pul = pd.DataFrame(righe)

for d in ["Sporchi", "Puliti"]:
    sub = df_pul[df_pul["dati"] == d]
    plt.plot(sub["livello"], sub["f1"], marker="o", label=d)
plt.title("Etichette sbagliate: F1 con dati sporchi vs puliti (SVM)")
plt.xlabel("livello"); plt.ylabel("F1"); plt.ylim(0, 1.02); plt.legend(); plt.show()
display(df_pul.round(3))

## 2. Outliers (valori anomali)

*Dimensione di qualita' violata: accuratezza sintattica.*

Sostituisco alcune celle con valori estremi (media +/- 6 deviazioni standard), come se fossero errori di
misura o di inserimento. Mi aspetto che il Decision Tree regga meglio dell'SVM: l'albero divide in base
all'ordine dei valori, mentre l'SVM risente di piu' dei numeri fuori scala.


In [ ]:
df_out = prova_su_livelli(rumore_outlier, LIVELLI, "livello")
display(df_out.round(3))
grafico(df_out, "livello", "Effetto degli outlier (test pulito)")

**Pulizia:** uso il *clipping* (winsorizzazione). Per ogni colonna calcolo il 10 e il 90 percentile e
riporto entro questi limiti i valori che li superano. Cosi' gli outlier vengono "schiacciati" a valori
ragionevoli senza perdere righe. La regola dei 3 sigma qui non funziona: gli outlier stessi gonfiano la
deviazione standard, quindi la soglia si allarga e non li intercetta piu'.


In [ ]:
LV = 0.20
X_sporco, y_sporco = rumore_outlier(LV)

# Senza pulizia
m = crea_svm(); m.fit(X_sporco, y_sporco)
prima = valuta(m); prima["dati"] = "Con outlier"

# Con pulizia: "taglio" i valori estremi entro il 10 e il 90 percentile di ogni colonna (clipping)
basso = np.percentile(X_sporco, 10, axis=0)
alto = np.percentile(X_sporco, 90, axis=0)
X_pulito = np.clip(X_sporco, basso, alto)
m = crea_svm(); m.fit(X_pulito, y_sporco)
dopo = valuta(m); dopo["dati"] = "Valori estremi tagliati (clipping)"

df_out_pul = pd.DataFrame([prima, dopo])
display(df_out_pul.round(3))
df_out_pul.set_index("dati")[["accuracy", "recall", "f1"]].plot(
    kind="bar", figsize=(8, 4.5), title="Outlier al 20%: prima e dopo il clipping (SVM)")
plt.ylim(0, 1.02); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3. Missing values (valori mancanti)

*Dimensione di qualita' violata: completezza.*

Metto dei NaN (valori mancanti) in una parte delle celle. I modelli non accettano i NaN, quindi come
gestione di base li riempio con la media della colonna (lo fa gia' la pipeline). La curva mostra che, anche
riempiendo, troppi buchi peggiorano i risultati.


In [ ]:
df_miss = prova_su_livelli(rumore_missing, LIVELLI, "livello")
display(df_miss.round(3))
grafico(df_miss, "livello", "Effetto dei valori mancanti (riempiti con la media)")

**Pulizia:** confronto tre modi di gestire i valori mancanti (al 20%):
- **cancellazione**: elimino le righe che hanno almeno un valore mancante;
- **media**: riempio i buchi con la media della colonna;
- **mediana**: riempio i buchi con la mediana della colonna.

La cancellazione e' pericolosa: con i NaN sparsi, quasi tutte le righe ne hanno almeno uno e rischio di
buttare via il dataset.


In [ ]:
LV = 0.20
X_sporco, y_sporco = rumore_missing(LV)
righe = []

# Cancellazione: tengo solo le righe senza alcun NaN
tieni = ~np.isnan(X_sporco).any(axis=1)
print("Cancellazione: rimangono", int(tieni.sum()), "righe su", len(y_sporco))
for nome, crea in MODELLI.items():
    if tieni.sum() >= 20:
        m = crea(); m.fit(X_sporco[tieni], y_sporco[tieni])
        righe.append({"strategia": "Cancellazione", "modello": nome, **valuta(m)})
    else:
        righe.append({"strategia": "Cancellazione (troppe poche righe)", "modello": nome,
                      "accuracy": np.nan, "recall": np.nan, "f1": np.nan})

# Media e mediana
for strategia in ["mean", "median"]:
    for nome, crea in MODELLI.items():
        passi = [("riempi", SimpleImputer(strategy=strategia))]
        if nome == "SVM":
            passi += [("scala", StandardScaler()), ("modello", SVC(kernel="rbf", random_state=SEED))]
        else:
            passi += [("modello", DecisionTreeClassifier(max_depth=3, random_state=SEED))]
        m = Pipeline(passi); m.fit(X_sporco, y_sporco)
        nome_str = "Media" if strategia == "mean" else "Mediana"
        righe.append({"strategia": nome_str, "modello": nome, **valuta(m)})

df_miss_pul = pd.DataFrame(righe)
display(df_miss_pul.round(3))
df_miss_pul.pivot_table(index="strategia", columns="modello", values="f1").plot(
    kind="bar", figsize=(9, 4.5), title="Missing al 20%: F1 per strategia di gestione")
plt.ylabel("F1"); plt.ylim(0, 1.02); plt.xticks(rotation=12); plt.tight_layout(); plt.show()

## 4. Overlapping (classi che si sovrappongono)

*Dimensione di qualita' violata: consistency.*

Aggiungo rumore gaussiano alle feature: piu' rumore metto (sigma piu' grande), piu' i punti delle due
classi si mescolano e diventa difficile separarli. Uso sigma = 0.5, 1.0, 1.5, 2.0.


In [ ]:
SIGMI = [0.0, 0.5, 1.0, 1.5, 2.0]
df_ov = prova_su_livelli(rumore_overlap, SIGMI, "sigma")
display(df_ov.round(3))
grafico(df_ov, "sigma", "Effetto della sovrapposizione delle classi (rumore sulle feature)")

# Mostro la sovrapposizione che cresce, proiettando i dati in 2D con la PCA
fig, assi = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, s in zip(assi, [0.0, 1.0, 2.0]):
    X_sporco, y_sporco = rumore_overlap(s)
    Z = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(X_sporco))
    ax.scatter(Z[y_sporco == 0, 0], Z[y_sporco == 0, 1], c="blue", s=10, alpha=0.6, label="Benigno")
    ax.scatter(Z[y_sporco == 1, 0], Z[y_sporco == 1, 1], c="red", s=10, alpha=0.6, label="Maligno")
    ax.set_title("sigma = " + str(s)); ax.legend()
fig.suptitle("Le due classi si sovrappongono sempre di piu")
plt.tight_layout(); plt.show()

**Pulizia:** uso la **PCA**. Proiettando i dati sulle prime componenti principali tengo le direzioni
con piu' informazione e scarto quelle con poca varianza, dove si nasconde parte del rumore. Confronto l'SVM
con e senza PCA.


In [ ]:
def crea_svm_pca(k=6):
    return Pipeline([("riempi", SimpleImputer(strategy="mean")), ("scala", StandardScaler()),
                     ("pca", PCA(n_components=k, random_state=SEED)),
                     ("modello", SVC(kernel="rbf", random_state=SEED))])

righe = []
for s in SIGMI:
    X_sporco, y_sporco = rumore_overlap(s)
    for nome, m in [("SVM senza PCA", crea_svm()), ("SVM con PCA", crea_svm_pca())]:
        m.fit(X_sporco, y_sporco)
        righe.append({"sigma": s, "metodo": nome, **valuta(m)})
df_ov_pul = pd.DataFrame(righe)

for nome in ["SVM senza PCA", "SVM con PCA"]:
    sub = df_ov_pul[df_ov_pul["metodo"] == nome]
    plt.plot(sub["sigma"], sub["f1"], marker="o", label=nome)
plt.title("Overlapping: la PCA aiuta a ridurre il rumore (F1)")
plt.xlabel("sigma"); plt.ylabel("F1"); plt.ylim(0, 1.02); plt.legend(); plt.show()
display(df_ov_pul.round(3))

## 5. Estensione: clustering non supervisionato (K-Means)

Finora ho usato modelli **supervisionati** (conoscono le etichette). Provo ora un modello **non
supervisionato**: il K-Means raggruppa i pazienti in 2 cluster **senza guardare la diagnosi**. Poi, solo per
valutare, confronto i cluster con le diagnosi vere con due metriche:
- **ARI** (Adjusted Rand Index): quanto i cluster coincidono con le classi vere (1 = perfetto, 0 = casuale);
- **Silhouette**: quanto i cluster sono compatti e ben separati.

Aggiungo rumore gaussiano crescente (lo stesso dell'esperimento overlapping) e guardo come peggiorano.

In [ ]:
# === CLUSTERING K-MEANS SOTTO RUMORE ===
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

righe = []
for s in SIGMI:
    X_sporco, y_sporco = rumore_overlap(s)
    X_scalato = StandardScaler().fit_transform(X_sporco)
    km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(X_scalato)
    righe.append({"sigma": s,
                  "ARI": adjusted_rand_score(y_sporco, km.labels_),
                  "silhouette": silhouette_score(X_scalato, km.labels_)})
df_clu = pd.DataFrame(righe)
display(df_clu.round(3))

plt.plot(df_clu["sigma"], df_clu["ARI"], marker="o", label="ARI (accordo con la diagnosi vera)")
plt.plot(df_clu["sigma"], df_clu["silhouette"], marker="s", label="Silhouette (compattezza cluster)")
plt.title("K-Means: il rumore peggiora la separazione tra le classi")
plt.xlabel("sigma rumore"); plt.ylabel("punteggio"); plt.ylim(0, 1.02); plt.legend(); plt.show()

## Riepilogo e conclusioni

Riassumo per ogni tipo di errore l'F1 (sui Maligni) con dati puliti e con il livello massimo di rumore.


In [ ]:
def riepilogo(df, colonna, nome_errore):
    pulito = df[df[colonna] == df[colonna].min()]
    sporco = df[df[colonna] == df[colonna].max()]
    out = []
    for nome in MODELLI:
        f1_pulito = pulito[pulito["modello"] == nome]["f1"].mean()
        f1_sporco = sporco[sporco["modello"] == nome]["f1"].mean()
        out.append({"Errore": nome_errore, "Modello": nome,
                    "F1 pulito": round(f1_pulito, 3), "F1 max rumore": round(f1_sporco, 3),
                    "Calo": round(f1_pulito - f1_sporco, 3)})
    return out

tabella = []
tabella += riepilogo(df_label, "livello", "Noisy labels")
tabella += riepilogo(df_out, "livello", "Outliers")
tabella += riepilogo(df_miss, "livello", "Missing values")
tabella += riepilogo(df_ov, "sigma", "Overlapping")
display(pd.DataFrame(tabella))

### Cosa ho osservato

- **Etichette sbagliate**: e' l'errore che fa piu' danno, perche' confonde direttamente il modello. Pulire
  i dati prima del training recupera una buona parte delle performance.
- **Outlier**: il Decision Tree regge meglio dell'SVM; il clipping (taglio dei valori estremi) recupera l'SVM.
- **Missing values**: con media o mediana il calo e' contenuto, mentre cancellare le righe e' rischioso
  quando i valori mancanti sono sparsi.
- **Overlapping**: peggiora gradualmente entrambi i modelli; la PCA aiuta perche' toglie le componenti con
  poca informazione, dove si concentra il rumore.

In generale: **la qualita' dei dati conta quanto il modello**. Anche un buon algoritmo, se addestrato su
dati sporchi, da' risultati peggiori; una pulizia adeguata dei dati permette di recuperare gran parte delle
performance.

**Nota sulle tecniche di preprocessing.** Tra le strategie anti-overfitting, *dropout* ed *early stopping*
non si applicano qui: riguardano le reti neurali, mentre uso Decision Tree e SVM. Per questi modelli le
strategie corrispondenti sono la *potatura* (pruning) dell'albero, la regolarizzazione (parametro C
dell'SVM) e la **cross-validation** usata sopra.

**Trade-off e ordine delle operazioni (slide 136-137).** Le dimensioni di qualita' sono spesso in trade-off
tra loro e, come visto a lezione (Learn2Clean), anche l'*ordine* con cui si applicano le operazioni di
pulizia influenza il risultato finale del modello.
